# 84 · Pathways a segment-level analysis puts in one segment: where they sit inside it on the elastic curve

A segment-resolution study calls pathways "enriched in PTS3" (or PTS1, PTS2). At that resolution they are one class.
This notebook asks where those pathways sit inside the segment along the package's elastic curve (first, middle or
last third), and whether they differ more than random groups of the same segment's genes.

**Protocol.** `docs/results/pt-within-segment-pathways.md` (frozen in 0281964; amended before any fit in 8b6cbe7:
the elastic curve is the only coordinate).

| Step | What | Data |
|---|---|---|
| D1–D4 | The segment-level call: moderated t per segment, cameraPR pathways, PT-specific core genes, representatives | Chen et al. 2021 microdissected mouse PTS1/PTS2/PTS3 |
| C1–C4 | Where each pathway sits inside the segment; one test per segment against a stratified gene permutation | 71's NB specimen curves on the elastic curve, 72's segment units |
| C5 | The same for the human sections (descriptive) | as above |

**What bounds the reading.**
- C14 (notebook 74): the elastic curve's fold noise exceeds the within-segment spread in 10 of 12 cells; only mouse
  S3 is not noise-dominated. Every within-segment statement is descriptive.
- Cohort: two male control mice and two cortex sections of one male human donor.
- The pathways come from mouse microdissection, so the human arm is descriptive.

## 1 · Setup

In [ ]:
import argparse
import hashlib
import json
import os
import re
import sys
import time
from pathlib import Path

for _var in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS'):
    os.environ.setdefault(_var, os.environ.get('PSEUDOSPACE_N_JOBS', '24'))   # thread cap; never in a cache key
NOTEBOOK_START = time.perf_counter()
starts = [Path.cwd()] + ([Path(__file__).resolve().parent] if '__file__' in globals() else [])
project = next(p for s in starts for p in (s, *s.parents) if (p / 'pseudospace').is_dir())
sys.path.insert(0, str(project))
parser = argparse.ArgumentParser(add_help=False)
parser.add_argument('--data-root', type=Path)
parser.add_argument('--results-root', type=Path)
args, _ = parser.parse_known_args()
data_root = Path(args.data_root or os.environ.get('PSEUDOSPACE_DATA_ROOT') or project / 'data').expanduser()
results_root = Path(args.results_root or os.environ.get('PSEUDOSPACE_RESULTS_ROOT') or project / 'results').expanduser()

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

EXPORT = results_root / 'package_pseudospace' / 'hvm' / 'pt_units.h5ad'
RUN = results_root / 'package_pseudospace' / 'runs' / 'hvm'
upstream13 = results_root / 'minimal_pt_scfates'                  # the ortholog map
T71 = results_root / 'pt_pathways_package_pseudospace'            # read only: universe, sets, stage cache
T72 = results_root / 'pt_package_coordinate_validation'           # read only: transitions
CHEN = data_root / 'external' / 'segment_microdissection' / 'GSE150338_Mouse_TPM_Replicates.txt.gz'
output = results_root / 'pt_within_segment_pathways'
tables, figures = output / 'tables', output / 'figures'
for folder in (tables, figures):
    folder.mkdir(parents=True, exist_ok=True)
DS = 'human_vs_mouse'
CHEN_SHA = 'daf8b6f46bf772bbbc950b68d688b7540528e2ec52b357440fac1ebe224d5488'
REQUIRED = [EXPORT, CHEN, upstream13 / 'ortholog_map_used.csv', T72 / 'tables' / 'transitions.csv',
            RUN / '03_integration' / 'pass1_labelled.h5ad', RUN / '03_integration' / 'pass2.h5ad',
            *[T71 / 'tables' / f for f in ('inputs.csv', f'universe_{DS}.csv', f'gene_sets_{DS}.json', f'gene_statistics_{DS}.csv')]]
for path in REQUIRED:
    if not path.is_file():
        raise FileNotFoundError(f'Missing input: {path} (run notebooks 70–72 first; see data/README.md)')

NOTEBOOK_LOGIC_VERSION = '84.within_segment_pathways.1'   # Bump when a calculation changes (no stage cache here).
FINGERPRINT = 'efafc9ac2e99'                              # confirmed hvm pass-1 labels (notebook 70)
SEGMENTS = ('S1', 'S2', 'S3')
NAMES = ['Ctrl1A2', 'Ctrl1A4', 'HUK1_COR1', 'HUK1_MED1']
MOUSE, HUMAN = NAMES[:2], NAMES[2:]
LABEL_ORIENTATION = {'Cyp2e1', 'Cyp7b1', 'Gatm', 'Slc7a13', 'Slc13a3', 'Slc22a6', 'Slc5a2', 'Slc22a7', 'Slc5a12',
                     'Aqp2', 'Aqp4', 'Atp6v0d2', 'Rhcg', 'Slc14a2', 'Wnt7b'}     # the 15 panel and anchor genes
NON_PT = ['DTL1', 'DTL2', 'DTL3', 'ATL', 'MTAL', 'CTAL', 'DCT', 'CNT', 'CCD', 'OMCD', 'IMCD', 'GL']
MIN_TPM, CAMERA_COR, FDR = 1., .01, .05                     # D1, D2
CORE_T, CORE_EFFECT, MIN_CORE, OVERLAP = 2., .5, 5, .6     # D3, D4
MIN_DETECTION, FLAT = .05, .25                             # 81's usable rule; C2
N_PERM, SEED, ALPHA = 2000, 84, .05                        # C3

MM = 1 / 25.4
INK, LIGHT = '#0b0b0b', '#d9d8d4'
SPEC_COLOR = {'Ctrl1A2': '#0072B2', 'Ctrl1A4': '#56B4E9', 'HUK1_COR1': '#D55E00', 'HUK1_MED1': '#E69F00'}
plt.rcParams.update({
    'font.family': ['Liberation Sans', 'DejaVu Sans'], 'font.size': 6.5, 'axes.titlesize': 6.5, 'axes.titleweight': 'bold',
    'axes.labelsize': 6.5, 'xtick.labelsize': 6, 'ytick.labelsize': 6, 'legend.frameon': False,
    'axes.spines.top': False, 'axes.spines.right': False, 'axes.linewidth': .6, 'lines.linewidth': 1.0,
    'pdf.fonttype': 42, 'svg.fonttype': 'none', 'savefig.dpi': 300})
pd.set_option('display.width', 240, 'display.max_columns', 40, 'display.max_colwidth', 70, 'display.max_rows', 200)
READING, GUARDS = [], []


def save(fig, stem):
    for suffix in ('png', 'pdf'):
        fig.savefig(figures / f'{stem}.{suffix}', bbox_inches='tight')
    plt.show()
    plt.close(fig)


def stored(folder, stage):
    """A notebook's newest cached payload of one stage (read only); raises when its cache was purged."""
    files = sorted((folder / 'stage_cache').glob(f'{stage}__*.npz'),
                   key=lambda p: json.loads(Path(str(p) + '.key.json').read_text())['written_utc'])
    if not files:
        raise FileNotFoundError(f'{folder.name}: no cached {stage}')
    with np.load(files[-1], allow_pickle=False) as payload:
        return {k: payload[k] for k in payload.files}


def short(pathway):
    lib, name = pathway.split('::', 1)
    name = re.sub(r'\s+R-HSA-\d+$', '', name)
    return f"{ {'Reactome_2022': 'R', 'KEGG_2019_Mouse': 'K', 'MSigDB_Hallmark_2020': 'H'}[lib] }: {name}"


print('Results folder:', output, '· logic', NOTEBOOK_LOGIC_VERSION)

## 2 · Inputs and guards (not cached)

- **G1:** the Chen file's sha256.
- **G2:** `hvm_inputs` reproduces 71's universe, sets and units; 71's cached condition model reproduces its T_spatial.
- **G3:** the logistic transitions reproduce 72's.

In [ ]:
import anndata as ad

from pseudospace.continuous_descriptors import logistic_transitions
from pseudospace.package_inputs import hvm_inputs, read_pt_units
from pseudospace.pathway_pipelines import camera_pr, common_grid
from pseudospace import published_findings as pf
from pseudospace import species_calls as sc
from pseudospace import within_segment as ws

t0 = time.perf_counter()
assert hashlib.sha256(CHEN.read_bytes()).hexdigest() == CHEN_SHA, 'G1: the Chen file differs from its sha256'
GUARDS.append('G1: Chen 2021 TPM file matches its sha256')

orthologs = pd.read_csv(upstream13 / 'ortholog_map_used.csv')
MT = sorted(g for g in orthologs.mouse_symbol.astype(str) if g.startswith('mt-'))
assert len(MT) == 13, MT
UNITS = read_pt_units(EXPORT, 'hvm')
PROV = UNITS.uns['provenance']
assert str(PROV['notebook_logic_version']).startswith('70.'), PROV['notebook_logic_version']
assert PROV['fingerprints']['pass1']['membership_sha1'] == FINGERPRINT, PROV['fingerprints']['pass1']
INP = hvm_inputs(UNITS, data_root, orthologs, exclude_genes=MT)
GENES = list(INP['genes'])
GIDX = {g: i for i, g in enumerate(GENES)}
SETS = INP['gene_sets']
universe71 = pd.read_csv(T71 / 'tables' / f'universe_{DS}.csv')
assert GENES == universe71[universe71.eligible.astype(bool)].gene.astype(str).tolist(), 'G2: universe differs from 71'
assert SETS == json.loads((T71 / 'tables' / f'gene_sets_{DS}.json').read_text()), 'G2: gene sets differ from 71'
assert int(pd.read_csv(T71 / 'tables' / 'inputs.csv', index_col=0).loc['units in the common support', 'Human vs mouse PT']) == len(INP['position'])
POSITION = np.asarray(INP['position'], float)
SPECIMEN = INP['specimen'].astype(str)
COUNTS = INP['counts'].tocsr()
STAT71 = pd.read_csv(T71 / 'tables' / f'gene_statistics_{DS}.csv', header=[0, 1], index_col=0)
COND = stored(T71, f'gene_model_{DS}_species')
np.testing.assert_allclose(COND['stat|T_spatial'], STAT71['condition'].T_spatial.to_numpy(), rtol=1e-12, atol=0)
GRID = COND['grid']
np.testing.assert_allclose(GRID, common_grid(POSITION, SPECIMEN, 101), rtol=0, atol=1e-12)
GUARDS.append(f'G2: {len(GENES):,} genes, {len(SETS):,} sets and {len(POSITION):,} units equal 71; '
              "71's cached condition model reproduces its T_spatial")

TRANSITIONS = pd.read_csv(T72 / 'tables' / 'transitions.csv')
TRANSITIONS = TRANSITIONS[TRANSITIONS.dataset.eq('hvm')].reset_index(drop=True)
again_t = logistic_transitions(POSITION, INP['segment'], SPECIMEN).merge(TRANSITIONS, on=['specimen', 'transition'])
assert len(again_t) == 8 and np.allclose(again_t.position_x, again_t.position_y, atol=1e-6)
CUTS = sc.mean_transitions(TRANSITIONS, NAMES)
UNITS_S = sc.to_segment_units(GRID, CUTS)                      # the curve grid in segment units
GUARDS.append(f"G3: logistic transitions reproduce 72's; cuts {CUTS[0]:.3f}, {CUTS[1]:.3f}; grid {UNITS_S[0]:.2f}–{UNITS_S[-1]:.2f}")

DETECTION = {spec: pd.Series(np.asarray((COUNTS[SPECIMEN == spec] > 0).mean(axis=0)).ravel(), index=GENES) for spec in NAMES}
USABLE = {spec: (COND[f'converged|{spec}'].astype(bool) & ~COND[f'separated|{spec}'].astype(bool)
                 & (DETECTION[spec].to_numpy() >= MIN_DETECTION)) for spec in NAMES}
REFERENCE = {spec: pf.reference_curve(COND[f'curve|{spec}'], USABLE[spec]) for spec in NAMES}
mouse_rows = np.isin(SPECIMEN, MOUSE)
LOG_MEAN_MOUSE = pd.Series(np.log(np.asarray(COUNTS[mouse_rows].mean(axis=0)).ravel() + 1e-9), index=GENES)

INTEGRATION = set()
for name in ('pass1_labelled.h5ad', 'pass2.h5ad'):
    obj = ad.read_h5ad(RUN / '03_integration' / name, backed='r')
    INTEGRATION |= set(obj.var_names[obj.var['highly_variable_for_harmony'].to_numpy(bool)].astype(str))
    obj.file.close()
BUILT = INTEGRATION | LABEL_ORIENTATION
GUARDS.append(f'Integration features {len(INTEGRATION):,} ({len(INTEGRATION & set(GENES)):,} in the universe) plus the 15 '
              'panel and anchor genes: the genes that built the curve')
print('\n'.join('· ' + g for g in GUARDS))
print(f'Inputs {time.perf_counter() - t0:.0f} s')

## 3 · D · The segment-level call (Chen 2021)

D1: log2(TPM + 1), technical replicates averaged per mouse × segment, moderated t of each segment against the mean of
the other two. D2: cameraPR of 71's sets on that t; Up and FDR ≤ 0.05. D3: cores (t ≥ 2, effect ≥ 0.5, PT-specific,
usable in both mice; ≥ 5). D4: representatives (core overlap ≥ 0.6 absorbs).

In [ ]:
raw = pd.read_csv(CHEN, sep='\t', compression='gzip')
pattern = re.compile(r'^m(\d+)_([A-Z0-9]+)_([a-c])$')
samples = pd.DataFrame([{'column': c, 'mouse': m.group(1), 'structure': m.group(2)}
                        for c in raw.columns if (m := pattern.match(str(c)))])
pt_cols = samples[samples.structure.isin(['PTS1', 'PTS2', 'PTS3'])]
assert pt_cols.groupby('structure').size().to_dict() == {'PTS1': 5, 'PTS2': 8, 'PTS3': 6}, pt_cols.groupby('structure').size()
assert set(NON_PT) <= set(samples.structure), set(NON_PT) - set(samples.structure)
tpm = raw.set_index('Gene_symbol')[samples.column.tolist()].astype(float)
tpm = tpm[tpm.index.notna()]
pt_mean = tpm[pt_cols.column].mean(axis=1)
tpm = tpm.assign(_m=pt_mean.to_numpy()).sort_values('_m', ascending=False)
tpm = tpm[~tpm.index.duplicated(keep='first')].drop(columns='_m')
seg_tpm = pd.DataFrame({s: tpm[samples.column[samples.structure.eq(s)]].mean(axis=1) for s in ['PTS1', 'PTS2', 'PTS3', *NON_PT]})
CHEN_GENES = [g for g in GENES if g in tpm.index and seg_tpm.loc[g, ['PTS1', 'PTS2', 'PTS3']].max() >= MIN_TPM]
log_pt = np.log2(tpm.loc[CHEN_GENES, pt_cols.column] + 1)
level = pt_cols.assign(key=pt_cols.structure + '|' + pt_cols.mouse)
mouse_level = log_pt.T.groupby(level.set_index('column').key.reindex(log_pt.columns).to_numpy()).mean().T
groups = np.array([k.split('|')[0].replace('PT', '') for k in mouse_level.columns])
assert dict(zip(*np.unique(groups, return_counts=True))) == {'S1': 3, 'S2': 5, 'S3': 3}
CHEN_T = {}
for seg in SEGMENTS:
    CHEN_T[seg] = ws.microdissection_segment_t(mouse_level, groups, seg, covariate=mouse_level.mean(axis=1))
pt_specific = {seg: seg_tpm.loc[CHEN_GENES, f'PT{seg}'] > seg_tpm.loc[CHEN_GENES, NON_PT].max(axis=1) for seg in SEGMENTS}
usable_mouse = pd.Series(USABLE[MOUSE[0]] & USABLE[MOUSE[1]], index=GENES)
usable_human = pd.Series(USABLE[HUMAN[0]] & USABLE[HUMAN[1]], index=GENES)
GUARDS.append(f'Chen: {len(CHEN_GENES):,} universe genes with mean TPM ≥ 1 in a PT segment; samples per mouse × '
              f'segment {dict(zip(*np.unique(groups, return_counts=True)))}')
print(GUARDS[-1])


def pool_of(seg, *, t_min=CORE_T, pt_filter=True, exclude=frozenset()):
    tab = CHEN_T[seg]
    ok = (tab.t >= t_min) & (tab.log2_effect >= CORE_EFFECT)
    if pt_filter:
        ok &= pt_specific[seg]
    genes = [g for g in ok.index[ok] if usable_mouse[g] and g not in exclude]
    return genes


CAMERA = {seg: camera_pr(CHEN_T[seg].t, SETS, inter_gene_cor=CAMERA_COR, fdr=FDR).set_index('pathway_id') for seg in SEGMENTS}


def segment_design(seg, **kw):
    """Called pathways, their cores (≥ MIN_CORE), and representatives for one segment under one core rule."""
    pool = pool_of(seg, **kw)
    pool_set = set(pool)
    called = CAMERA[seg][CAMERA[seg].call].sort_values(['p']).index.tolist()
    called = sorted(called, key=lambda p: (CAMERA[seg].loc[p, 'p'], p))
    cores = {p: sorted(set(SETS[p]) & pool_set) for p in called}
    cores = {p: c for p, c in cores.items() if len(c) >= MIN_CORE}
    reps = ws.greedy_representatives(cores, list(cores), overlap=OVERLAP)
    return {'pool': pool, 'called': called, 'cores': cores, 'reps': reps}


DESIGN = {seg: segment_design(seg) for seg in SEGMENTS}
rows = []
for seg in SEGMENTS:
    d = DESIGN[seg]
    theme_of = {p: r for r, absorbed in d['reps'].items() for p in [r, *absorbed]}
    for p in d['called']:
        rows.append({'segment': seg, 'pathway': p, 'name': short(p), 'n': int(CAMERA[seg].loc[p, 'n']),
                     'p': CAMERA[seg].loc[p, 'p'], 'fdr': CAMERA[seg].loc[p, 'fdr'],
                     'core': len(d['cores'].get(p, [])), 'core genes': ', '.join(d['cores'].get(p, [])),
                     'representative': theme_of.get(p, '') == p, 'theme': short(theme_of[p]) if p in theme_of else ''})
SEGMENT_PATHWAYS = pd.DataFrame(rows)
summary = pd.DataFrame({seg: {'called (Up, FDR ≤ 0.05)': len(DESIGN[seg]['called']), 'pool genes': len(DESIGN[seg]['pool']),
                              'with ≥ 5 core genes': len(DESIGN[seg]['cores']), 'representatives': len(DESIGN[seg]['reps'])}
                        for seg in SEGMENTS}).T
display(summary)
display(SEGMENT_PATHWAYS[SEGMENT_PATHWAYS.representative][['segment', 'name', 'n', 'fdr', 'core']].round(4))

## 4 · C · Where each pathway sits inside the segment (elastic curve)

C1: per specimen, gene curves are 71's NB curves centred by their mean and by the specimen's median usable gene
(`pf.reference_curve`); a pathway curve is the median over its core (log2). C2: thirds profile inside the segment.
C3: T and its stratified permutation p (expression tertile × integration feature; Holm over S1–S3). C4: reportable
positions and the fold-protected column.

In [ ]:
def gene_matrix(genes, spec):
    """Genes x grid of centred gene curves (log2); NaN rows where the gene is not usable in ``spec``."""
    idx = np.array([GIDX[g] for g in genes], int)
    y = COND[f'curve|{spec}'][idx].astype(float)
    y = (y - y.mean(axis=1, keepdims=True) - REFERENCE[spec]) / np.log(2)
    y[~USABLE[spec][idx]] = np.nan
    return y


def strata_of(genes, stratified=True):
    if not stratified:
        return np.zeros(len(genes), int)
    tert = pd.qcut(LOG_MEAN_MOUSE.reindex(genes).rank(method='first'), 3, labels=False).to_numpy()
    built = np.array([g in BUILT for g in genes], int)
    return tert * 2 + built


def run_segment(seg, design, specs, *, stratified=True, n_perm=N_PERM):
    k = SEGMENTS.index(seg)
    reps = list(design['reps'])
    pool = design['pool']
    if len(reps) < 3:
        return {'T': np.nan, 'p': np.nan, 'testable': False, 'reps': reps}
    pidx = {g: i for i, g in enumerate(pool)}
    cores = [np.array([pidx[g] for g in design['cores'][r]], int) for r in reps]
    a, b = (gene_matrix(pool, s) for s in specs)
    res = ws.position_test(a, b, UNITS_S, cores, k, strata_of(pool, stratified), n_perm=n_perm, seed=SEED)
    res.update({'testable': True, 'reps': reps})
    return res


TESTS = {}
for seg in SEGMENTS:
    TESTS[('mouse', seg)] = run_segment(seg, DESIGN[seg], MOUSE)
p_raw = pd.Series({s: TESTS[('mouse', s)]['p'] for s in SEGMENTS}).dropna().sort_values()
holm, running = {}, 0.
for rank, (seg, p) in enumerate(p_raw.items()):
    running = max(running, min(1., (len(p_raw) - rank) * p))
    holm[seg] = running
for seg in SEGMENTS:
    TESTS[('mouse', seg)]['p_holm'] = holm.get(seg, np.nan)
print(f'Tests {time.perf_counter() - t0:.0f} s')

In [ ]:
def pathway_rows(seg, design, species, specs, label='primary'):
    k = SEGMENTS.index(seg)
    out = []
    for r in design['reps']:
        core = design['cores'][r]
        fp = [g for g in core if g not in BUILT]
        row = {'segment': seg, 'species': species, 'arm': label, 'pathway': r, 'name': short(r), 'core': len(core),
               'core outside integration features': len(fp)}
        prof = {}
        for spec in specs:
            curve = np.nanmedian(gene_matrix(core, spec), axis=0)
            prof[spec] = ws.thirds_profile(curve, UNITS_S, k)
            row.update({f'{spec} first': prof[spec]['profile'][0], f'{spec} middle': prof[spec]['profile'][1],
                        f'{spec} last': prof[spec]['profile'][2], f'{spec} range': prof[spec]['range'],
                        f'{spec} peak position': prof[spec]['peak position'],
                        f'{spec} peak third': prof[spec]['peak part']})
        a, b = (prof[s] for s in specs)
        agree = (not a['flat']) and (not b['flat']) and a['peak part'] == b['peak part']
        row['reportable'] = bool(agree)
        row['peak third'] = ['first', 'middle', 'last'][a['peak part']] if agree else ''
        row['peak position (mean)'] = (a['peak position'] + b['peak position']) / 2
        if len(fp) >= MIN_CORE:
            pf_prof = [ws.thirds_profile(np.nanmedian(gene_matrix(fp, s), axis=0), UNITS_S, k) for s in specs]
            row['fold-protected'] = bool(agree and not pf_prof[0]['flat'] and not pf_prof[1]['flat']
                                         and pf_prof[0]['peak part'] == pf_prof[1]['peak part'] == a['peak part'])
        else:
            row['fold-protected'] = 'not evaluable'
        out.append(row)
    return out


POSITIONS = pd.DataFrame([r for seg in SEGMENTS for r in pathway_rows(seg, DESIGN[seg], 'mouse', MOUSE)])
test_rows = []
for seg in SEGMENTS:
    res = TESTS[('mouse', seg)]
    sub = POSITIONS[POSITIONS.segment.eq(seg) & POSITIONS.reportable]
    thirds = sorted(set(sub['peak third']))
    claim = bool(res['testable'] and res['p_holm'] <= ALPHA and len(thirds) >= 2)
    test_rows.append({'species': 'mouse', 'segment': seg, 'representatives': len(res['reps']), 'T': res['T'],
                      'null median': float(np.median(res['null'])) if res['testable'] else np.nan,
                      'p': res['p'], 'p Holm': res['p_holm'], 'reportable': len(sub),
                      'reportable thirds': ', '.join(f"{t} {int((sub['peak third'] == t).sum())}" for t in thirds),
                      'statement (descriptive under C14)': 'pathways separate inside the segment' if claim else
                      ('not testable' if not res['testable'] else 'no within-segment separation')})
TEST_TABLE = pd.DataFrame(test_rows)
display(TEST_TABLE.round(4))
display(POSITIONS[['segment', 'name', 'core', 'core outside integration features', 'Ctrl1A2 peak position',
                   'Ctrl1A4 peak position', 'Ctrl1A2 range', 'Ctrl1A4 range', 'peak third', 'reportable',
                   'fold-protected']].round(2))

## 5 · C5 · Human sections (descriptive)

The same representatives on the two human sections; genes must be usable in both. No claim.

In [ ]:
def human_design(seg):
    d = DESIGN[seg]
    cores = {r: [g for g in d['cores'][r] if usable_human[g]] for r in d['reps']}
    cores = {r: c for r, c in cores.items() if len(c) >= MIN_CORE}
    return {'pool': [g for g in d['pool'] if usable_human[g]], 'cores': cores, 'reps': {r: [] for r in cores}}


HUMAN_DESIGN = {seg: human_design(seg) for seg in SEGMENTS}
for seg in SEGMENTS:
    TESTS[('human', seg)] = run_segment(seg, HUMAN_DESIGN[seg], HUMAN)
HUMAN_POSITIONS = pd.DataFrame([r for seg in SEGMENTS for r in pathway_rows(seg, HUMAN_DESIGN[seg], 'human', HUMAN)])
human_tests = pd.DataFrame([{'species': 'human', 'segment': seg, 'representatives': len(TESTS[('human', seg)]['reps']),
                             'T': TESTS[('human', seg)]['T'], 'p': TESTS[('human', seg)]['p']} for seg in SEGMENTS])
display(human_tests.round(4))
if len(HUMAN_POSITIONS):
    display(HUMAN_POSITIONS[['segment', 'name', 'core', 'HUK1_COR1 peak position', 'HUK1_MED1 peak position',
                             'peak third', 'reportable']].round(2))

## 6 · Sensitivities (descriptive)

In [ ]:
SENS = {'no PT-specific filter': {'pt_filter': False}, 'core t ≥ 3': {'t_min': 3.},
        'cores outside integration features (fold protection)': {'exclude': frozenset(BUILT)}}
sens_rows = []
for label, kw in SENS.items():
    for seg in SEGMENTS:
        d = segment_design(seg, **kw)
        res = run_segment(seg, d, MOUSE)
        pos = pd.DataFrame(pathway_rows(seg, d, 'mouse', MOUSE, label)) if res['testable'] else pd.DataFrame()
        sens_rows.append({'sensitivity': label, 'segment': seg, 'representatives': len(d['reps']), 'T': res['T'],
                          'p': res['p'], 'reportable': int(pos.reportable.sum()) if len(pos) else 0,
                          'reportable thirds': ', '.join(sorted(set(pos.loc[pos.reportable, 'peak third']))) if len(pos) else ''})
for seg in SEGMENTS:
    res = run_segment(seg, DESIGN[seg], MOUSE, stratified=False)
    sens_rows.append({'sensitivity': 'unstratified permutation', 'segment': seg, 'representatives': len(DESIGN[seg]['reps']),
                      'T': res['T'], 'p': res['p']})
SENSITIVITIES = pd.DataFrame(sens_rows)
display(SENSITIVITIES.round(4))
ALL_PATHWAYS = pd.DataFrame([r for seg in SEGMENTS
                             for r in pathway_rows(seg, {**DESIGN[seg], 'reps': {p: [] for p in DESIGN[seg]['cores']}},
                                                   'mouse', MOUSE, 'every pathway')])
print(f'Sensitivities {time.perf_counter() - t0:.0f} s')

## 7 · Figures

- **84a–c** (one per segment): left, Chen's segment means of each representative's core (median over core genes of
  centred log2 TPM); right, the mouse pathway curves (mean of the two mice) along the elastic curve, rows ordered by
  peak position inside the segment, a dot at each row's peak (filled when reportable).
- **84d:** representatives with reportable positions in different thirds (earliest, middle, latest), one line per
  specimen.

In [ ]:
from matplotlib.colors import TwoSlopeNorm


def segment_figure(seg, stem):
    k = SEGMENTS.index(seg)
    d = DESIGN[seg]
    pos = POSITIONS[POSITIONS.segment.eq(seg)].set_index('pathway')
    if pos.empty:
        print(f'{seg}: no representative; no figure')
        return
    reps = pos.sort_values('peak position (mean)').index.tolist()
    chen = np.array([mouse_level.loc[d['cores'][r]].T.groupby(groups).mean().T.pipe(lambda m: m.sub(m.mean(axis=1), axis=0))
                     .median().reindex(list(SEGMENTS)).to_numpy() for r in reps])
    curves = np.array([np.mean([np.nanmedian(gene_matrix(d['cores'][r], s), axis=0) for s in MOUSE], axis=0) for r in reps])
    height = max(30, 4.2 * len(reps) + 14) * MM
    fig, (left, right) = plt.subplots(1, 2, figsize=(170 * MM, height), gridspec_kw={'width_ratios': [1, 4.2], 'wspace': .04})
    lim = max(np.nanmax(np.abs(chen)), .5)
    left.imshow(chen, cmap='RdBu_r', norm=TwoSlopeNorm(0, -lim, lim), aspect='auto')
    left.set_xticks(range(3))
    left.set_xticklabels(['PTS1', 'PTS2', 'PTS3'])
    left.set_yticks(range(len(reps)))
    left.set_yticklabels([short(r) for r in reps])
    lim2 = max(np.nanmax(np.abs(curves)), .25)
    right.imshow(curves, cmap='RdBu_r', norm=TwoSlopeNorm(0, -lim2, lim2), aspect='auto',
                 extent=(UNITS_S[0], UNITS_S[-1], len(reps) - .5, -.5))
    for b in (1, 2):
        right.axvline(b, color=INK, lw=.6)
    for i, r in enumerate(reps):
        right.scatter([pos.loc[r, 'peak position (mean)']], [i], s=8, color=INK if pos.loc[r, 'reportable'] else 'white',
                      edgecolors=INK, linewidths=.5, zorder=3)
    right.set_yticks([])
    right.set_xlim(0, 3)
    right.set_xlabel('elastic curve (segment units)')
    save(fig, stem)


for seg, stem in zip(SEGMENTS, ['fig84a_s1_pathways', 'fig84b_s2_pathways', 'fig84c_s3_pathways']):
    segment_figure(seg, stem)


def examples_figure(stem):
    picks = []
    for seg in reversed(SEGMENTS):                       # S3 first: the one cell that is not noise-dominated
        sub = POSITIONS[POSITIONS.segment.eq(seg) & POSITIONS.reportable].sort_values('peak position (mean)')
        if sub['peak third'].nunique() >= 2:
            chosen = [sub.iloc[0], sub.iloc[len(sub) // 2], sub.iloc[-1]] if len(sub) >= 3 else [sub.iloc[0], sub.iloc[-1]]
            seen, picks = set(), []
            for row in chosen:
                if row.pathway not in seen:
                    picks.append(row)
                    seen.add(row.pathway)
            break
    if not picks:
        print('No segment has reportable positions in two thirds; 84d not drawn')
        return
    seg = picks[0].segment
    k = SEGMENTS.index(seg)
    fig, axes = plt.subplots(1, len(picks), figsize=(60 * MM * len(picks), 42 * MM), sharey=True, squeeze=False)
    for ax, row in zip(axes[0], picks):
        core = DESIGN[seg]['cores'][row.pathway]
        ax.axvspan(k, k + 1, color='#f2f1ee', zorder=0)
        for spec in NAMES:
            ax.plot(UNITS_S, np.nanmedian(gene_matrix(core, spec), axis=0), color=SPEC_COLOR[spec], lw=.9)
        for b in (1, 2):
            ax.axvline(b, color=LIGHT, lw=.6)
        ax.axhline(0, color=LIGHT, lw=.5)
        ax.set_title(row['name'], loc='left')
        ax.set_xlim(0, 3)
        ax.set_xlabel('elastic curve (segment units)')
    axes[0, 0].set_ylabel('log2, centred')
    save(fig, stem)


examples_figure('fig84d_examples')

## 8 · Outputs and run record

In [ ]:
chen_out = pd.concat({seg: CHEN_T[seg][['log2_effect', 't', 'p']] for seg in SEGMENTS}, axis=1)
chen_out[('PT-specific', 'S1')] = pt_specific['S1']
chen_out[('PT-specific', 'S2')] = pt_specific['S2']
chen_out[('PT-specific', 'S3')] = pt_specific['S3']
chen_out.to_csv(tables / 'chen_segment_t.csv')
SEGMENT_PATHWAYS.to_csv(tables / 'segment_pathways.csv', index=False)
pd.concat([POSITIONS, HUMAN_POSITIONS, ALL_PATHWAYS], ignore_index=True).to_csv(tables / 'positions.csv', index=False)
pd.concat([TEST_TABLE, human_tests], ignore_index=True).to_csv(tables / 'tests.csv', index=False)
SENSITIVITIES.to_csv(tables / 'sensitivities.csv', index=False)
READING.extend([
    f"Logic version {NOTEBOOK_LOGIC_VERSION!r}, as the protocol names it.",
    'D1: the Chen columns m{mouse}_{structure}_{a,b,c} are the replicate samples; the bare structure columns at the end '
    'of the file are summaries and are not used.',
    'D3: "usable in both mice" is 81\'s rule (71 curve converged and not separated, detection ≥ 5% over all of the '
    "specimen's units).",
    'C3: Holm over the three mouse segments; a segment that is not testable takes no part in the correction.',
    'C5: a human core keeps the mouse core genes usable in both human sections; a representative needs ≥ 5 of them.',
    'Sensitivities run the same 2,000 permutations (seed 84).',
])
pd.Series(READING, name='how the protocol was read').to_csv(tables / 'how_the_protocol_was_read.csv', index=False)
record = {'notebook_logic_version': NOTEBOOK_LOGIC_VERSION,
          'protocol': 'docs/results/pt-within-segment-pathways.md (0281964, amended 8b6cbe7)',
          'pass1_fingerprint': FINGERPRINT, 'chen_sha256': CHEN_SHA, 'guards': GUARDS,
          'design': summary.to_dict('index'), 'tests': TEST_TABLE.to_dict('records'), 'human_tests': human_tests.to_dict('records'),
          'sensitivities': SENSITIVITIES.to_dict('records'), 'seconds': round(time.perf_counter() - NOTEBOOK_START, 1)}
(output / 'run_record.json').write_text(json.dumps(record, indent=1, default=str))
for line in READING:
    print('·', line)
print(f"Run record written · {record['seconds']} s")